In [ ]:
import geopandas as gpd
from shapely.ops import unary_union
from shapely.geometry import MultiLineString
import pandas as pd

# creating the bikepath polygon - represents 300m of every bikepath on the city

# get all bikepaths
bikepaths_GDF = gpd.read_file('./data/Ciclorrotas.shp')
bikelanes_GDF = gpd.read_file('./data/Ciclovias.shp')
today_bikelanes_GDF = pd.concat([bikepaths_GDF, bikelanes_GDF])

# clear gdfs from multilinestrings, they stop later processes
multilinestring_rows = today_bikelanes_GDF.geometry.apply(lambda geom: isinstance(geom, MultiLineString))
today_bikelanes_GDF = today_bikelanes_GDF[~multilinestring_rows]
today_bikelanes_GDF = today_bikelanes_GDF.reset_index()

# switch crs modes, create 300m buffer around all bikepaths
today_bikelanes_GDF = today_bikelanes_GDF.set_crs(epsg=4674)
today_bikelanes_GDF = today_bikelanes_GDF.to_crs(epsg=31983) # web mercator, 1:1 com metros
today_bikelanes_GDF_bufferzone = gpd.GeoDataFrame(geometry=[])
today_bikelanes_GDF_bufferzone['geometry'] = today_bikelanes_GDF.geometry.buffer(300)
today_bikelanes_GDF_bufferzone = today_bikelanes_GDF_bufferzone.set_crs(epsg=31983)
today_bikelanes_GDF_bufferzone = today_bikelanes_GDF_bufferzone.to_crs(epsg=4674)

# make buffer polygons into one single entity
today_bikelanes_merged_polygon = unary_union(today_bikelanes_GDF_bufferzone.geometry)

GDF_saopaulo = gpd.read_file('./data/sao_paulo_demographics.geojson')

sao_paulo_subprefeituras_gdf = gpd.read_file('./data/SIRGAS_SHP_subprefeitura_polygon.shp')
# Given the zones in alphabetical order, the 'Zona' list assign them to the cardinal zone (related by hand)  

# Subprefeitura = sao_paulo_subprefeituras_gdf['sp_nome'].sort_values()

# Subprefeitura = [
#     'ARICANDUVA-FORMOSA-CARRAO', 'BUTANTA', 'CAMPO LIMPO', 'CAPELA DO SOCORRO',
#     'CASA VERDE-CACHOEIRINHA', 'CIDADE ADEMAR', 'CIDADE TIRADENTES', 'ERMELINO MATARAZZO',
#     'FREGUESIA-BRASILANDIA', 'GUAIANASES', 'IPIRANGA', 'ITAIM PAULISTA', 'ITAQUERA',
#     'JABAQUARA', 'JACANA-TREMEMBE', 'LAPA', 'M BOI MIRIM', 'MOOCA', 'PARELHEIROS', 'PENHA',
#     'PERUS', 'PINHEIROS', 'PIRITUBA-JARAGUA', 'SANTANA-TUCURUVI', 'SANTO AMARO',
#     'SAO MATEUS', 'SAO MIGUEL', 'SAPOPEMBA', 'SE', 'VILA MARIA-VILA GUILHERME',
#     'VILA MARIANA', 'VILA PRUDENTE'
# ]

zona = [
    'Leste', 'Oeste', 'Sul', 'Sul',
    'Norte', 'Sul', 'Leste', 'Leste',
    'Norte', 'Leste', 'Sul', 'Leste', 'Leste',
    'Sul', 'Norte', 'Oeste', 'Sul', 'Leste', 'Sul', 'Leste',
    'Norte', 'Oeste', 'Norte', 'Norte', 'Sul',
    'Leste', 'Leste', 'Leste', 'Central', 'Norte',
    'Sul', 'Leste'
]
# making separate polygons for each zone and assigning crs 

sao_paulo_subprefeituras_gdf = sao_paulo_subprefeituras_gdf.sort_values(by='sp_nome')
sao_paulo_subprefeituras_gdf['sp_nome'] = zona
sao_paulo_subprefeituras_gdf = sao_paulo_subprefeituras_gdf.set_crs(epsg=31983)

GDF_sp_subpref_bikelanes = gpd.overlay(sao_paulo_subprefeituras_gdf, today_bikelanes_GDF_bufferzone.to_crs(epsg=31983), keep_geom_type=False, how="intersection")
GDF_sp_subpref_bikelanes = GDF_sp_subpref_bikelanes.dissolve(by='sp_nome', as_index=False)

GDF_sp_subpref_bufferzones = gpd.overlay(GDF_sp_subpref_bikelanes, GDF_saopaulo, keep_geom_type=False, how="intersection")
GDF_saopaulo_filtered = GDF_saopaulo[GDF_saopaulo['id'].isin(GDF_sp_subpref_bufferzones['id'])]
GDF_sp_subpref_bufferzones['area1'] = GDF_sp_subpref_bufferzones.area
GDF_saopaulo_filtered['area2'] = GDF_saopaulo_filtered.area
GDF_merge = GDF_sp_subpref_bufferzones[['id', 'area1']].merge(
    GDF_saopaulo_filtered[['id', 'area2']],)
GDF_sp_subpref_bufferzones['area_ratio'] = GDF_merge['area1']/GDF_merge['area2']
# Por alguma razão, a abordagem GDF_intersection_zones_gdfmerge['areas_ratio'] = GDF_intersection_zones_gdfmerge.to_crs('EPSG:31983').area/GDF_saopaulo_overlaid_zones.to_crs('EPSG:31983').area
# não funciona, mas a abordagem GDF_sp_subpref_bufferzones['area_ratio'] = GDF_merge['area1']/GDF_merge['area2'] funciona perfeitamente
GDF_sp_subpref_bufferzones = GDF_sp_subpref_bufferzones.reset_index(drop=True)
GDF_sp_subpref_bufferzones['estimate_population'] = GDF_sp_subpref_bufferzones['populacao'] * GDF_sp_subpref_bufferzones['area_ratio']
GDF_zones_numbers = GDF_sp_subpref_bufferzones[['sp_nome', 'estimate_population']]
GDF_sp_subpref_zones = gpd.overlay(GDF_saopaulo, sao_paulo_subprefeituras_gdf, keep_geom_type=False, how="intersection")
GDF_sp_subpref_zones = GDF_sp_subpref_zones[~GDF_sp_subpref_zones.duplicated('id')]
GDF_sp_subpref_zones = GDF_sp_subpref_zones.groupby('sp_nome', as_index=False).agg({'populacao': 'sum', 'geometry': 'first'})
GDF_zones_numbers = GDF_zones_numbers.groupby('sp_nome', as_index=False).sum()
GDF_zones_numbers['population'] = GDF_sp_subpref_zones['populacao']
GDF_zones_numbers['PNB_score'] = GDF_zones_numbers['estimate_population'] / GDF_zones_numbers['population'] 
GDF_sp_subpref_bufferzones.to_file('./data/sao_paulo_bikelanes_subprefeituras.geojson', driver='GeoJSON')
GDF_zones_numbers.to_csv('./data/sao_paulo_zones_numbers.csv')

# # matching both gdfs order
# GDF_intersection_zones_gdfmerge = GDF_intersection_zones_gdfmerge.sort_values('id', ascending = True)
# GDF_saopaulo_overlaid_zones = GDF_saopaulo_overlaid_zones.sort_values('id', ascending = True)
# GDF_intersection_zones_gdfmerge = GDF_intersection_zones_gdfmerge.reset_index()
# GDF_saopaulo_overlaid_zones = GDF_saopaulo_overlaid_zones.reset_index()

# # calculating the estimate population in said intersected zones (accounting for parcial intersection)
# GDF_intersection_zones_gdfmerge['areas_ratio'] = GDF_intersection_zones_gdfmerge.to_crs('EPSG:31983').area/GDF_saopaulo_overlaid_zones.to_crs('EPSG:31983').area
# GDF_intersection_zones_gdfmerge['estimate_population'] = GDF_intersection_zones_gdfmerge['populacao'] * GDF_intersection_zones_gdfmerge['areas_ratio']

# GDF_intersection_zones_gdfmerge['estimate_population'].sum()

GDF_zones_numbers = GDF_sp_subpref_bufferzones[['sp_nome', 'estimate_population']]
GDF_sp_subpref_zones = gpd.overlay(GDF_saopaulo, sao_paulo_subprefeituras_gdf, keep_geom_type=False, how="intersection")
GDF_sp_subpref_zones = GDF_sp_subpref_zones[~GDF_sp_subpref_zones.duplicated('id')]
GDF_sp_subpref_zones = GDF_sp_subpref_zones.groupby('sp_nome', as_index=False).agg({'populacao': 'sum', 'geometry': 'first'})
GDF_zones_numbers = GDF_zones_numbers.groupby('sp_nome', as_index=False).sum()
GDF_zones_numbers['population'] = GDF_sp_subpref_zones['populacao']
GDF_zones_numbers['PNB_score'] = GDF_zones_numbers['estimate_population'] / GDF_zones_numbers['population'] 
estimate_pop_in_PNB = GDF_zones_numbers['estimate_population'].sum()
pop = GDF_zones_numbers['population'].sum()
total_row = {"sp_nome":"Total", "estimate_population":estimate_pop_in_PNB, "population":pop, "PNB_score":estimate_pop_in_PNB/pop}
GDF_zones_numbers.loc[len(GDF_zones_numbers)] = total_row

c:\Users\João Rahal\anaconda3\Lib\site-packages\geopandas\geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [ ]:
from shapely import LineString
from shapely.ops import unary_union
import folium
import json

file_name = ".//data//caminhos_bike_3000m_11.geojson"
with open(file_name, 'r', encoding='utf-8') as file:
    data = json.load(file)

fmap = folium.Map(location=[-23.5, -46.6], zoom_start=10)

i = 0
new_lanes_GDF = gpd.GeoDataFrame(geometry=[])

while i < len(data['features']):
    coordinates = data['features'][i]['properties']['paths'][0]['points']['coordinates']
    linestring_coordinates = LineString([(point[0], point[1]) for point in coordinates])
    temp_gdf = gpd.GeoDataFrame(geometry=[linestring_coordinates])
    new_lanes_GDF = pd.concat([new_lanes_GDF , temp_gdf], ignore_index=True)
    i += 1

#     folium.GeoJson(
#         data={
#             'type': 'Feature',
#             'geometry': linestring_coordinates.__geo_interface__
#         },
#         name='LineString',
#         style_function=lambda x: {'color': 'red'}
#     ).add_to(fmap)

# fmap

new_lanes_GDF = new_lanes_GDF.set_crs(epsg=4326)
new_lanes_GDF = new_lanes_GDF.to_crs(epsg=31983)

today_bikelanes_multilinestring = unary_union(today_bikelanes_GDF.geometry)
new_lanes_GDF = new_lanes_GDF.geometry.apply(lambda line: line.difference(today_bikelanes_multilinestring))

new_lanes_GDF_bufferzone = gpd.GeoDataFrame(geometry=[])
new_lanes_GDF_bufferzone['geometry'] = new_lanes_GDF.geometry.buffer(300)
new_lanes_GDF_bufferzone = new_lanes_GDF_bufferzone.set_crs(epsg=31983)
new_lanes_GDF_bufferzone = new_lanes_GDF_bufferzone.to_crs(epsg=4326)

theorized_bikelanes_merged_polygon = unary_union(new_lanes_GDF_bufferzone.geometry)

In [17]:
from shapely import union

today_and_theorized_polygon_merge = union(theorized_bikelanes_merged_polygon, today_bikelanes_merged_polygon)

# make the area polygon into a GDF and assign a crs for it 
GDF_today_and_theorized_polygon_merge = gpd.GeoDataFrame(geometry=[today_and_theorized_polygon_merge])
GDF_today_and_theorized_polygon_merge = GDF_today_and_theorized_polygon_merge.set_crs('EPSG:4326')
GDF_today_and_theorized_polygon_merge = GDF_today_and_theorized_polygon_merge.to_crs('EPSG:31983')

# finding census zones intersecting the bikelane polygon
GDF_intersection_zones_gdfmerge = gpd.overlay(GDF_saopaulo, GDF_today_and_theorized_polygon_merge, how="intersection")

# removing non-intersecting zones
GDF_saopaulo_overlaid_zones = GDF_saopaulo[GDF_saopaulo['id'].isin(GDF_intersection_zones_gdfmerge['id'])]

# matching both gdfs order
GDF_intersection_zones_gdfmerge = GDF_intersection_zones_gdfmerge.sort_values('id', ascending = True)
GDF_saopaulo_overlaid_zones = GDF_saopaulo_overlaid_zones.sort_values('id', ascending = True)
GDF_intersection_zones_gdfmerge = GDF_intersection_zones_gdfmerge.reset_index()
GDF_saopaulo_overlaid_zones = GDF_saopaulo_overlaid_zones.reset_index()

# calculating the estimate population in said intersected zones (accounting for parcial intersection)
GDF_intersection_zones_gdfmerge['areas_ratio'] = GDF_intersection_zones_gdfmerge.to_crs('EPSG:31983').area/GDF_saopaulo_overlaid_zones.to_crs('EPSG:31983').area
GDF_intersection_zones_gdfmerge['estimate_population'] = GDF_intersection_zones_gdfmerge['populacao'] * GDF_intersection_zones_gdfmerge['areas_ratio']

GDF_intersection_zones_gdfmerge['estimate_population'].sum()


4950829.223903104

# Estimativa atual inalterada (microdata de 2010):
##   3378449 (30.02%) pessoas à 300m de infraestrutura cicloviária

# Estimativa com rotas teorizadas (1000m 1:1):
##   4320607 (+942158) pessoas à 300m de infraestrutura cicloviária
##   (30.02% -> 38.39%, + 8.37%)
##   necessários 318km de ciclovias novas 

# Estimativa com rotas teorizadas (3000m 1:1):
##   4758654 (+1380205) pessoas à 300m de infraestrutura cicloviária
##   (30.02% -> 42.28%, + 12.26%)
##   necessários 864km de ciclovias novas

# Estimativa com rotas teorizadas (3000m 1:1):
##   5303848 (+1925399) pessoas à 300m de infraestrutura cicloviária
##   (30.02% -> 47.13%, + 17.11%)
##   necessários 1128km de ciclovias novas